# Fleet Capacity Optimization: Should Agnes Trucking Repair Its 15 Idle Tractors?

*Queen's University, Smith School of Business, MMA program | Team Agnes*

**Business question.** Agnes Trucking has 107 tractors, but 15 sit idle in maintenance. On busy days the overflow loads go to the spot market at $3.00 to $3.50 per mile, against an internal cost of about $0.93 per mile. Is a $2,000 repair per truck worth it?

**Approach.** Every weekly load is assigned either to one of the company's own trucks or to the spot market, so as to maximize profit. Two scenarios are compared:

1. **Baseline:** only the 92 active trucks are available.
2. **Repair:** the model may pay $2,000 to bring back any of the 15 idle trucks, and does so only where it pays off.

**Rules the model must follow.** Every load is covered once. A truck can carry at most one load on any given day. An idle truck can only be used if its repair fee is paid.

> **About this notebook.** A clean rebuild of the team's original Colab notebook (Pyomo with the open-source HiGHS solver), re-run end to end against the case workbook. The code is organized into one reusable function so both scenarios and the sensitivity tests use identical rules. Every scenario is solved to proven optimality.

## 1. Setup

In [1]:
# Uncomment on Google Colab (not needed if packages are already installed):
# !pip install -q pyomo highspy openpyxl pandas

import os
import pandas as pd
import pyomo.environ as pyo

pd.options.display.float_format = "{:,.2f}".format

## 2. Load the case data

The workbook holds the weekly loads, the truck list, and the brand-level operating cost per mile. Only trucks marked **Active** (92) or **Maintenance** (15) take part. The 13 **Inactive** trucks are outside the scope of the decision.

In [2]:
# Path to the case workbook. Adjust if you keep it elsewhere.
candidates = ["../data/Project_Agnes.xlsx", "data/Project_Agnes.xlsx", "Project_Agnes.xlsx"]
XLS = next(p for p in candidates if os.path.exists(p))
DATA_DIR = os.path.dirname(XLS) or "."

loads = pd.read_excel(XLS, sheet_name="SAMPLE DATA").dropna(subset=["load_id"])
trucks = pd.read_excel(XLS, sheet_name="Trucks")
brand_cost = pd.read_excel(XLS, sheet_name="Variable Cost").iloc[:6]   # one row per truck brand

loads["load_date"] = pd.to_datetime(loads["load_date"])
loads["Load_close date"] = pd.to_datetime(loads["Load_close date"])

print(f"{len(loads)} weekly loads | {loads['load_date'].min().date()} to {loads['Load_close date'].max().date()}")
print(trucks["status"].value_counts().to_string())

377 weekly loads | 2024-01-01 to 2024-01-10
status
Active         92
Maintenance    15
Inactive       13


## 3. Cost per mile

Each brand has its own fuel and maintenance cost per mile (from the *Variable Cost* sheet). Spot-market freight is priced by trailer type.

In [3]:
cost_per_mile_by_brand = dict(zip(brand_cost["Truck Brand"], brand_cost["Total Variable Cost"]))
SPOT_RATE = {"Dry Van": 3.00, "Refrigerated": 3.50}   # dollars per mile
REPAIR_FEE = 2000                                      # one-time, per idle truck

display(brand_cost.rename(columns={"Fleet Count": "Trucks", "Total Variable Cost": "Cost per mile ($)"})
        [["Truck Brand", "Trucks", "Cost per mile ($)"]].style.format({"Cost per mile ($)": "{:.3f}"}))

,Truck Brand,Trucks,Cost per mile ($)
0,Volvo,20.000000,0.928
1,Kenworth,15.000000,0.951
2,International,18.000000,0.951
3,Freightliner,22.000000,0.955
4,Mack,21.000000,0.958
5,Peterbilt,24.000000,0.961


## 4. The optimization model

One function builds and solves either scenario. For each load it chooses an own truck or the spot market, with profit = revenue minus (miles × cost per mile). The function also takes the repair fee and a spot-rate multiplier so the sensitivity tests in Section 6 reuse exactly the same rules.

In [4]:
active_ids = trucks.loc[trucks["status"] == "Active", "truck_id"].tolist()
maint_ids  = trucks.loc[trucks["status"] == "Maintenance", "truck_id"].tolist()
make_of = trucks.set_index("truck_id")["make"].to_dict()
cpm = {k: cost_per_mile_by_brand[make_of[k]] for k in active_ids + maint_ids}

L = loads.set_index("load_id")
LOADS = L.index.tolist()
dist, rev, ltype = L["Distance"].to_dict(), L["Total Revenue"].to_dict(), L["load_type"].to_dict()

# Which loads are in transit on each calendar day? A truck can carry only one of them.
loads_on_day = {}
for l in LOADS:
    for d in pd.date_range(L.at[l, "load_date"], L.at[l, "Load_close date"]):
        loads_on_day.setdefault(d, []).append(l)


def solve_fleet(allow_repair, repair_fee=REPAIR_FEE, spot_mult=1.0, time_limit=300):
    """Maximize weekly profit. allow_repair=False is the baseline (92 active trucks)."""
    idle = maint_ids if allow_repair else []
    fleet = active_ids + idle
    spot = {l: SPOT_RATE[ltype[l]] * spot_mult for l in LOADS}

    m = pyo.ConcreteModel()
    m.x = pyo.Var(LOADS, fleet, domain=pyo.Binary)      # load l on own truck k
    m.y = pyo.Var(LOADS, domain=pyo.Binary)             # load l sent to spot market
    if idle:
        m.z = pyo.Var(idle, domain=pyo.Binary)          # pay to repair idle truck k

    own  = pyo.quicksum((rev[l] - dist[l] * cpm[k]) * m.x[l, k] for l in LOADS for k in fleet)
    spot_p = pyo.quicksum((rev[l] - dist[l] * spot[l]) * m.y[l] for l in LOADS)
    fees = pyo.quicksum(repair_fee * m.z[k] for k in idle) if idle else 0
    m.obj = pyo.Objective(expr=own + spot_p - fees, sense=pyo.maximize)

    # 1. Every load is covered exactly once
    m.cover = pyo.Constraint(LOADS, rule=lambda m, l: sum(m.x[l, k] for k in fleet) + m.y[l] == 1)
    # 2. An idle truck can be used only if it is repaired
    if idle:
        m.link = pyo.Constraint(LOADS, idle, rule=lambda m, l, k: m.x[l, k] <= m.z[k])
    # 3. One load per truck per day
    m.day = pyo.ConstraintList()
    for d, ls in loads_on_day.items():
        if len(ls) > 1:
            for k in fleet:
                m.day.add(sum(m.x[l, k] for l in ls) <= 1)

    res = pyo.SolverFactory("appsi_highs")
    res.config.time_limit = time_limit
    out = res.solve(m)
    assert out.solver.termination_condition == pyo.TerminationCondition.optimal, f"Solver did not reach optimality: {out.solver.termination_condition}"

    assign = {}
    for l in LOADS:
        k = next((k for k in fleet if m.x[l, k].value > 0.5), "Spot Market")
        assign[l] = k
    own_profit  = sum((rev[l] - dist[l] * cpm[k]) for l, k in assign.items() if k != "Spot Market")
    spot_profit = sum((rev[l] - dist[l] * spot[l]) for l, k in assign.items() if k == "Spot Market")
    repaired = [k for k in idle if m.z[k].value > 0.5]
    return dict(net_profit=pyo.value(m.obj), own_profit=own_profit, spot_profit=spot_profit,
                repair_cost=repair_fee * len(repaired), repaired=repaired,
                spot_loads=sum(k == "Spot Market" for k in assign.values()), assign=assign)

## 5. Results: baseline versus repair

In [5]:
base = solve_fleet(allow_repair=False)
repair = solve_fleet(allow_repair=True)

summary = pd.DataFrame({
    "Baseline (92 trucks)": [92, 0, base["own_profit"], base["spot_profit"], base["net_profit"], base["spot_loads"]],
    "Repair (up to 107)":   [92 + len(repair["repaired"]), repair["repair_cost"], repair["own_profit"],
                             repair["spot_profit"], repair["net_profit"], repair["spot_loads"]],
}, index=["Active trucks", "One-time repair cost ($)", "Profit on own trucks ($)",
          "Profit (loss) on spot market ($)", "Net weekly profit ($)", "Loads sent to spot market"])
summary["Change"] = summary.iloc[:, 1] - summary.iloc[:, 0]
display(summary)

gain = repair["net_profit"] - base["net_profit"]
print(f"Trucks repaired: {len(repair['repaired'])} of {len(maint_ids)}")
print(f"Net profit gain in the modelled week (after the repair cost): ${gain:,.2f}")

,Baseline (92 trucks),Repair (up to 107),Change
Active trucks,92.00,107.00,15.00
One-time repair cost ($),0.00,"30,000.00","30,000.00"
Profit on own trucks ($),"490,881.87","547,356.64","56,474.77"
Profit (loss) on spot market ($),"-150,660.73","-126,155.89","24,504.84"
Net weekly profit ($),"340,221.14","391,200.75","50,979.61"
Loads sent to spot market,203.00,181.00,-22.00


Trucks repaired: 15 of 15
Net profit gain in the modelled week (after the repair cost): $50,979.61


**Reading the result.** Repairing the idle trucks moves loads off the spot market, where the company loses money on every mile, and onto its own trucks. The repair fee is recovered within the first couple of loads each repaired truck carries.

In [6]:
# Payback per truck: miles needed for one repaired truck to earn back its fee.
# Each mile moved from spot to an own truck saves (spot rate - own cost). Using the load-mix average:
avg_spot = sum(SPOT_RATE[ltype[l]] * dist[l] for l in LOADS) / sum(dist.values())
avg_own  = sum(cpm.values()) / len(cpm)
print(f"Average saving per mile moved off the spot market: ${avg_spot - avg_own:,.2f}")
print(f"Break-even per repaired truck: about {REPAIR_FEE / (avg_spot - avg_own):,.0f} miles "
      f"(typical loads run 500 to 2,000+ miles)")

Average saving per mile moved off the spot market: $2.29
Break-even per repaired truck: about 873 miles (typical loads run 500 to 2,000+ miles)


## 6. How robust is the decision?

Two inputs are uncertain: what the repair really costs, and what the spot market charges. Each is varied while everything else is held fixed.

### 6.1 Repair cost

In [7]:
rows = []
gross = repair["net_profit"] + repair["repair_cost"] - base["net_profit"]      # savings before the fee
breakeven_fee = gross / len(maint_ids)
for fee in [2000, 3500, 5000, 5250, 5500]:
    r = solve_fleet(True, repair_fee=fee)
    rows.append([fee, len(r["repaired"]), r["net_profit"], r["net_profit"] - base["net_profit"]])
sens_fee = pd.DataFrame(rows, columns=["Repair fee per truck ($)", "Trucks repaired", "Net weekly profit ($)", "Gain vs baseline ($)"])
display(sens_fee)
print(f"Break-even repair fee: ${breakeven_fee:,.2f} per truck. Above this, the model chooses not to repair.")

,Repair fee per truck ($),Trucks repaired,Net weekly profit ($),Gain vs baseline ($)
0,2000,15,"391,200.75","50,979.61"
1,3500,15,"368,700.75","28,479.61"
2,5000,15,"346,200.75","5,979.61"
3,5250,12,"342,494.09","2,272.96"
4,5500,5,"340,746.68",525.54


Break-even repair fee: $5,398.64 per truck. Above this, the model chooses not to repair.


### 6.2 Spot-market rates (±20%)

In [8]:
rows = []
for mult, label in [(0.8, "-20%"), (1.0, "Base"), (1.2, "+20%")]:
    b = base if mult == 1.0 else solve_fleet(False, spot_mult=mult)
    r = repair if mult == 1.0 else solve_fleet(True, spot_mult=mult)
    rows.append([label, b["net_profit"], r["net_profit"], r["net_profit"] - b["net_profit"], len(r["repaired"])])
sens_spot = pd.DataFrame(rows, columns=["Spot rates", "Baseline profit ($)", "Repair profit ($)", "Benefit of repair ($)", "Trucks repaired"])
display(sens_spot)

,Spot rates,Baseline profit ($),Repair profit ($),Benefit of repair ($),Trucks repaired
0,-20%,"473,660.12","501,463.52","27,803.40",15
1,Base,"340,221.14","391,200.75","50,979.61",15
2,+20%,"206,813.33","281,017.18","74,203.85",15


## 7. Which trucks to repair, and the load-by-load schedule

The schedule below is what a dispatcher would use: each load is either given to a named truck or sent to the spot market. It is saved to the data folder as an Excel file.

In [9]:
def schedule(res):
    out = loads[["load_id", "route_id", "load_type", "Distance", "Total Revenue", "load_date", "Load_close date"]].copy()
    out["Assigned to"] = out["load_id"].map(res["assign"])
    out["Fulfilled by"] = out["Assigned to"].map(lambda a: "Spot market" if a == "Spot Market" else "Own truck")
    return out

sched_base, sched_repair = schedule(base), schedule(repair)
print("Trucks approved for repair:", ", ".join(repair["repaired"]))
display(sched_repair.head(8))

out_path = os.path.join(DATA_DIR, "Optimized_Fleet_Schedule.xlsx")
with pd.ExcelWriter(out_path) as xw:
    sched_base.to_excel(xw, sheet_name="Baseline (92 trucks)", index=False)
    sched_repair.to_excel(xw, sheet_name="After repair", index=False)
print("Saved:", out_path)

Trucks approved for repair: TRK00003, TRK00005, TRK00017, TRK00027, TRK00028, TRK00032, TRK00034, TRK00037, TRK00042, TRK00047, TRK00066, TRK00077, TRK00087, TRK00104, TRK00109


,load_id,route_id,load_type,Distance,Total Revenue,load_date,Load_close date,Assigned to,Fulfilled by
0,LOAD00056755,RTE00007,Refrigerated,1576,"4,127.75",2024-01-01,2024-01-03,TRK00112,Own truck
1,LOAD00056756,RTE00030,Refrigerated,2147,"5,569.37",2024-01-01,2024-01-04,TRK00106,Own truck
2,LOAD00056757,RTE00048,Dry Van,2332,"6,943.50",2024-01-01,2024-01-05,Spot Market,Spot market
3,LOAD00056758,RTE00006,Dry Van,675,"1,493.85",2024-01-01,2024-01-02,TRK00054,Own truck
4,LOAD00056759,RTE00041,Dry Van,1216,"3,516.04",2024-01-01,2024-01-03,TRK00009,Own truck
5,LOAD00056760,RTE00001,Refrigerated,677,"1,424.77",2024-01-01,2024-01-02,TRK00078,Own truck
6,LOAD00056761,RTE00045,Dry Van,597,"1,734.47",2024-01-01,2024-01-02,TRK00049,Own truck
7,LOAD00056762,RTE00027,Refrigerated,1271,"3,195.79",2024-01-01,2024-01-03,TRK00067,Own truck


Saved: ../data/Optimized_Fleet_Schedule.xlsx


## 8. Conclusion and limits

**Recommendation.** Approve the $30,000 repair budget for all 15 idle tractors. The decision holds across repair costs up to roughly $5,000 per truck and across ±20% moves in spot-market rates.

**Limits.**
- Results are for one representative week. Scaling to a year assumes similar demand in other weeks.
- Trucks are assumed able to start from any location, so repositioning (empty) miles are not charged. Real savings will be somewhat lower.
- Repaired trucks are assumed to stay in service all week, with drivers available.
- The model treats every truck as able to haul every load. The workbook has no trailer-type data per truck, so trailer matching is not modelled here.